# MORE Solar Park: preprocessing to phase A

This notebook cleans the raw CSVs and writes the phase-A outputs.


In [1]:
import gzip
import warnings
warnings.filterwarnings("ignore")
import json
import os
import platform
import random
import sys
import hashlib

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = "."
OUT_DIR = "phase_a"
os.makedirs(OUT_DIR, exist_ok=True)

LOCAL_TZ = "Europe/Athens"
STEP_MIN = 15
SITE_LAT, SITE_LON = 40.5, 21.8
LABEL_SHIFT_MIN = 7.5
NIGHT_ELEV, DAY_ELEV = -6.0, 10.0
WSPD_MAX, TEMP_MIN, IRR_MAX = 40.0, -30.0, 1400.0
STUCK_MIN_ROWS = 12
TEMP_RECOVERY_MIN_C = -2.0
IRR_FAULT_NIGHT_MEDIAN = 8.0
OFFSET_WINDOW_DAYS, OFFSET_MIN_DAYS = 7, 3
FFILL_LIMIT, IRR_FFILL_LIMIT = 4, 2
ZERO_IRR, ZERO_PU = 100.0, 0.02
ARR_MEDIAN_MIN, ARR_MIN_RUN = 0.3, 8
PURGE_ROWS = 192
WARMUP_ROWS = 1344


def runs(mask, min_len=1):
    m = mask.fillna(False).astype(bool)
    grp = (m != m.shift()).cumsum()
    out = []
    for _, s in m[m].groupby(grp[m]):
        if len(s) >= min_len:
            out.append((s.index[0], s.index[-1], len(s)))
    return out


def identical_runs(s, min_rows):
    grp = (s != s.shift()).cumsum()
    return (s.groupby(grp).transform("size") >= min_rows) & s.notna()


def solar_elevation(t_utc, lat, lon):
    n = (pd.DatetimeIndex(t_utc) - pd.Timestamp("2000-01-01 12:00", tz="UTC")) / pd.Timedelta(days=1)
    n = np.asarray(n, dtype=float)
    L = np.deg2rad((280.460 + 0.9856474 * n) % 360)
    g = np.deg2rad((357.528 + 0.9856003 * n) % 360)
    lam = L + np.deg2rad(1.915) * np.sin(g) + np.deg2rad(0.020) * np.sin(2 * g)
    eps = np.deg2rad(23.439 - 0.0000004 * n)
    dec = np.arcsin(np.sin(eps) * np.sin(lam))
    ra = np.arctan2(np.cos(eps) * np.sin(lam), np.cos(lam))
    lst = np.deg2rad(((280.46061837 + 360.98564736629 * n) + lon) % 360)
    ha = lst - ra
    la = np.deg2rad(lat)
    return np.rad2deg(np.arcsin(np.sin(la) * np.sin(dec) + np.cos(la) * np.cos(dec) * np.cos(ha)))


def dump_json(obj, path):
    def default(o):
        if hasattr(o, "item"):
            return float(o.item())
        return str(o)
    with open(path, "w") as fh:
        json.dump(obj, fh, indent=2, default=default)

print({"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__, "seed": SEED})


{'python': '3.12.3', 'pandas': '3.1.0', 'numpy': '2.5.4', 'seed': 42}


## 1) Load raw CSVs
Weather data uses latin-1 encoding.


In [2]:
W = pd.read_csv(f"{DATA_DIR}/Weather_Data.csv", encoding="latin-1")
P = pd.read_csv(f"{DATA_DIR}/Plant_Active_Power.csv", encoding="latin-1")
A = pd.read_csv(f"{DATA_DIR}/Array_Power.csv", encoding="latin-1")
print("shapes:", W.shape, P.shape, A.shape)
assert (W["Timestamp"] == P["Timestamp"]).all() and (W["Timestamp"] == A["Timestamp"]).all(), "timestamps differ between files"

wcols = ["temp_c", "irr_wm2", "mod_temp_c", "precip_mmh", "rh_pct", "wdir_deg", "wspd_ms"]
arr_cols = [f"arr{i}" for i in range(1, 15)]
assert W.shape[1] == 8 and A.shape[1] == 15 and P.shape[1] == 2
naive = pd.to_datetime(W["Timestamp"], format="%d/%m/%y %H:%M") # day-first
W.columns = ["Timestamp"] + wcols
P.columns = ["Timestamp", "plant_pu"]
A.columns = ["Timestamp"] + arr_cols


shapes: (280512, 8) (280512, 2) (280512, 15)


## 2) Rebuild UTC
Local timestamps include DST. Rebuild UTC from row order and verify it two ways.


In [3]:
d_naive = naive.diff().dropna()
n_dup = int(naive.duplicated().sum())
n_spring = int((d_naive == pd.Timedelta("75min")).sum()) # missing hour
n_fall = int((d_naive == pd.Timedelta("-45min")).sum()) # repeated hour
print("duplicated naive stamps:", n_dup, "| spring jumps:", n_spring, "| fall repeats:", n_fall)

first_utc = naive.iloc[0].tz_localize(LOCAL_TZ).tz_convert("UTC")
idx = pd.date_range(first_utc, periods=len(W), freq=f"{STEP_MIN}min", name="time_utc")

 # local time check
match = float((idx.tz_convert(LOCAL_TZ).tz_localize(None) == pd.DatetimeIndex(naive)).mean())
 # last row check
last_expected = naive.iloc[-1].tz_localize(LOCAL_TZ).tz_convert("UTC")
print(f"rebuilt local == original in {match:.4%} of rows | last row matches: {idx[-1] == last_expected}")
assert match == 1.0 and idx[-1] == last_expected
for df_ in (W, P, A):
    df_.index = idx
raw = W[wcols].join(P[["plant_pu"]]).join(A[arr_cols])
assert raw.index.to_series().diff().dropna().eq(pd.Timedelta(f"{STEP_MIN}min")).all()
print(raw.shape, raw.index.min(), "->", raw.index.max())


duplicated naive stamps: 32 | spring jumps: 8 | fall repeats: 8
rebuilt local == original in 100.0000% of rows | last row matches: True
(280512, 22) 2015-12-31 22:15:00+00:00 -> 2023-12-31 22:00:00+00:00


## 3) Solar geometry
Compute solar elevation at interval midpoints; night is below -6 degrees. The table checks EET/EEST versus CET.


In [4]:
mid = idx - pd.Timedelta(minutes=LABEL_SHIFT_MIN)
elev = pd.Series(solar_elevation(mid, SITE_LAT, SITE_LON), index=idx, name="solar_elev_deg")

 # healthy period
healthy = raw.loc["2016-01-01":"2021-11-30", "irr_wm2"].dropna()
rows = []
for shift in (-60.0, -15.0, -7.5, 0.0, 7.5, 15.0, 60.0):
    for lat in (38.0, 40.5, 42.0):
        e = solar_elevation(healthy.index + pd.Timedelta(minutes=shift), lat, SITE_LON)
        viol = int(((e < NIGHT_ELEV) & (healthy.values > 2)).sum() + ((e > 4) & (healthy.values <= 0.5)).sum())
        rows.append({"label_shift_min": shift, "lat": lat, "violations": viol})
ev = pd.DataFrame(rows).pivot(index="label_shift_min", columns="lat", values="violations")
print("solar geometry violations: lower is better")
print(ev)
best_small = ev.loc[[-15.0, -7.5, 0.0, 7.5, 15.0]].min().min()
print("large shifts clearly worse (as expected):", ev.loc[[-60.0, 60.0]].min().min(), "vs", best_small)


solar geometry violations: lower is better
lat              38.0  40.5  42.0
label_shift_min                  
-60.0            6997  6690  6531
-15.0             364   371   392
-7.5              324   319   329
 0.0              336   329   376
 7.5              401   460   485
 15.0             619   651   849
 60.0            9973  9595  9397
large shifts clearly worse (as expected): 6531 vs 319


## 4) Audit raw data


In [5]:
audit = pd.DataFrame({
    "nulls": raw[wcols + ["plant_pu"]].isna().sum(),
    "null_%": (raw[wcols + ["plant_pu"]].isna().mean() * 100).round(2),
    "min": raw[wcols + ["plant_pu"]].min().round(3),
    "max": raw[wcols + ["plant_pu"]].max().round(3),
})
print(audit)
print("array columns null:", int(raw[arr_cols].isna().sum().sum()), "| plant range:", raw["plant_pu"].min(), raw["plant_pu"].max())
print("rows with ALL weather null:", int(raw[wcols].isna().all(axis=1).sum()))


            nulls  null_%     min       max
temp_c       4844    1.73 -39.118    41.479
irr_wm2      4869    1.74   0.000  1177.137
mod_temp_c   4796    1.71 -13.392    69.998
precip_mmh   4990    1.78   0.000    13.025
rh_pct       5468    1.95   8.999    99.860
wdir_deg     5452    1.94  26.105   401.598
wspd_ms      5447    1.94   0.000    89.035
plant_pu        0    0.00   0.000     0.918
array columns null: 0 | plant range: 0.0 0.918051842
rows with ALL weather null: 4794


## 5) Clean weather
Seven cleaning rules. Original values stay in the flags.


In [6]:
w = raw[wcols].copy()
raw_nan = w.isna()
rule_nan = pd.DataFrame(False, index=idx, columns=wcols)
orig_vals = {c: pd.Series(np.nan, index=idx) for c in ("temp_c", "irr_wm2", "mod_temp_c", "wspd_ms")}
rule_hits, rule_rows = {}, {}


def set_nan(col, mask, rule):
    m = mask.fillna(False) & w[col].notna()
    if col in orig_vals:
        orig_vals[col] = orig_vals[col].where(~m, w[col])
    w.loc[m, col] = np.nan
    rule_nan.loc[m, col] = True
    rule_hits[rule] = rule_hits.get(rule, 0) + int(m.sum())
    rule_rows[rule] = m if rule not in rule_rows else (rule_rows[rule] | m)
    return m


 # wind speed
set_nan("wspd_ms", w["wspd_ms"] > WSPD_MAX, "wspd_gt_40")

 # air temperature
bad_temp = w["temp_c"] < TEMP_MIN
ep_start = w.index[bad_temp].min()
dmin = w["temp_c"].resample("D").min()
back = dmin[(dmin.index > ep_start.normalize()) & (dmin > TEMP_RECOVERY_MIN_C)]
ep_end = back.index[0]
episode = pd.Series((w.index >= ep_start) & (w.index < ep_end), index=idx)
assert episode[bad_temp].all(), "temp < -30 rows outside the failure episode"
set_nan("temp_c", episode, "temp_sensor_episode")

 # stuck module temperature
stuck = identical_runs(w["mod_temp_c"], STUCK_MIN_ROWS)
set_nan("mod_temp_c", stuck, "modt_stuck")

 # wind direction
wrapped = w["wdir_deg"] > 360
w["wdir_deg"] = w["wdir_deg"] % 360
rule_hits["wdir_wrapped_mod360"] = int(wrapped.sum())

 # irradiance faults
is_night = elev < NIGHT_ELEV
is_day = elev > DAY_ELEV
night_irr = w["irr_wm2"].where(is_night)
day_key = idx.normalize()
all_days = pd.date_range(idx[0].normalize(), idx[-1].normalize(), freq="D")
dn = night_irr.groupby(day_key).median().where(night_irr.groupby(day_key).count() >= 8).reindex(all_days)
fault_days = dn.index[dn > IRR_FAULT_NIGHT_MEDIAN]
irr_fault = pd.Series(day_key.isin(fault_days), index=idx)
set_nan("irr_wm2", irr_fault, "irr_fault_window")
fault_windows = [(s.date(), e.date()) for s, e, _ in runs(pd.Series(dn > IRR_FAULT_NIGHT_MEDIAN, index=all_days))]

 # irradiance offset
dn_ok = dn.where(dn <= IRR_FAULT_NIGHT_MEDIAN)
offset_day = (dn_ok.shift(1).rolling(OFFSET_WINDOW_DAYS, min_periods=OFFSET_MIN_DAYS).median()
              .fillna(0.0).clip(0.0, IRR_FAULT_NIGHT_MEDIAN))
offset_row = pd.Series(offset_day.reindex(day_key).to_numpy(), index=idx, name="irr_offset_applied_wm2")
w["irr_wm2"] = (w["irr_wm2"] - offset_row).clip(lower=0.0)
off_rows = (offset_row >= 1.0) & ~irr_fault

 # physical bounds
set_nan("irr_wm2", w["irr_wm2"] > IRR_MAX, "irr_gt_1400")

resid = w["irr_wm2"][is_night & ~irr_fault].dropna()
resid_by_year = resid.groupby(resid.index.year).median().round(2).to_dict()
print("median night irradiance AFTER offset correction, by year:", resid_by_year)
assert max(resid_by_year.values()) <= 0.5, "night offset not removed"

print("episode (temp sensor):", ep_start, "->", ep_end)
print("irradiance fault windows (UTC days):", fault_windows)
print("offset period rows:", int(off_rows.sum()), "|", off_rows[off_rows].index.min(), "->", off_rows[off_rows].index.max())
print("rule hits:", rule_hits)


median night irradiance AFTER offset correction, by year: {2015: 0.0, 2016: 0.0, 2017: 0.0, 2018: 0.0, 2019: 0.0, 2020: 0.0, 2021: 0.0, 2022: 0.0, 2023: 0.0}
episode (temp sensor): 2022-03-22 14:30:00+00:00 -> 2022-03-26 00:00:00+00:00
irradiance fault windows (UTC days): [(datetime.date(2020, 5, 6), datetime.date(2020, 5, 6)), (datetime.date(2021, 12, 4), datetime.date(2022, 2, 22)), (datetime.date(2023, 10, 12), datetime.date(2023, 12, 31))]
offset period rows: 40032 | 2022-08-21 00:00:00+00:00 -> 2023-10-11 23:45:00+00:00
rule hits: {'wspd_gt_40': 2, 'temp_sensor_episode': 326, 'modt_stuck': 60, 'wdir_wrapped_mod360': 1597, 'irr_fault_window': 15591, 'irr_gt_1400': 0}


### Fill short gaps
Short gaps are forward-filled; longer gaps remain missing.


In [7]:
fill_ok = raw_nan & ~rule_nan
imputed = pd.DataFrame(False, index=idx, columns=wcols)
for c in wcols:
    lim = IRR_FFILL_LIMIT if c == "irr_wm2" else FFILL_LIMIT
    f = w[c].ffill(limit=lim)
    m = fill_ok[c] & w[c].isna() & f.notna()
    w.loc[m, c] = f[m]
    imputed[c] = m

rad = np.deg2rad(w["wdir_deg"])
w["wind_u"] = -w["wspd_ms"] * np.sin(rad)
w["wind_v"] = -w["wspd_ms"] * np.cos(rad)
weather_cols = wcols + ["wind_u", "wind_v"]

wx_imputed_count = imputed.sum(axis=1).astype(int)
wx_unfilled_any = w[wcols].isna().any(axis=1)
all_missing_raw = raw_nan.all(axis=1)
print("imputed values:", int(imputed.sum().sum()), "| rows still with any weather NaN:", int(wx_unfilled_any.sum()),
      "| rows with all weather missing (raw):", int(all_missing_raw.sum()))

 # consistency check
chk = (w["mod_temp_c"] - w["temp_c"])[is_day & (w["irr_wm2"] > 300)].dropna()
print("daytime module-ambient diff: max =", round(float(chk.max()), 1))
assert chk.max() <= 45


imputed values: 921 | rows still with any weather NaN: 21109 | rows with all weather missing (raw): 4794
daytime module-ambient diff: max = 37.6


## 6) Operational events
Keep real plant and array events as evidence for anomaly detection.


In [8]:
plant = raw["plant_pu"]
irr = w["irr_wm2"]
cand_zero = is_day & (irr > ZERO_IRR) & (plant < ZERO_PU)

arr_med = raw[arr_cols].median(axis=1)
arr_out = pd.DataFrame({c: (raw[c] == 0) & (arr_med > ARR_MEDIAN_MIN) & is_day for c in arr_cols})
arr_out_count = arr_out.sum(axis=1).astype(int)

 #
naive_rule = (raw["irr_wm2"] > 50) & (plant < 0.05)
print("plant-zero candidates (daytime, irr>100):", int(cand_zero.sum()), "in", len(runs(cand_zero)), "events")
print("naive rule (irr>50 & plant<0.05, no sun-elevation filter):", int(naive_rule.sum()), "rows")
print("array outage rows per array:", arr_out.sum().to_dict())

events = []
def add_event(kind, start, end, rows, detail, action):
    events.append({"type": kind, "start_utc": start, "end_utc": end, "rows": int(rows), "detail": detail, "action": action})

for s, e, n in runs(all_missing_raw, min_len=4):
    add_event("weather_all_missing", s, e, n, "all 7 weather sensors missing", "kept NaN (gap > fill limit)" if n > FFILL_LIMIT else "filled")
part = raw_nan.any(axis=1) & ~all_missing_raw
for s, e, n in runs(part, min_len=24):
    cols_ = raw_nan.loc[s:e].mean()
    add_event("weather_partial_missing", s, e, n, "missing: " + ",".join(cols_[cols_ > 0.5].index), "kept NaN (gap > fill limit)")
for s, e, n in runs(irr_fault):
    add_event("irradiance_sensor_fault", s, e, n, "night-median irradiance above %.0f W/m2" % IRR_FAULT_NIGHT_MEDIAN, "irradiance set to NaN; original in flags")
for s, e, n in runs(off_rows):
    add_event("irradiance_offset", s, e, n, "constant night offset ~%.1f W/m2" % float(offset_row[off_rows].median()), "causal offset subtracted")
add_event("temp_sensor_episode", ep_start, ep_end - pd.Timedelta(minutes=STEP_MIN), int(episode.sum()), "ambient temp ~ -39C then recovery", "temp set to NaN; original in flags")
for s, e, n in runs(stuck):
    add_event("module_temp_stuck", s, e, n, "constant module temperature", "set to NaN; original in flags")
for s, e, n in runs(cand_zero):
    allz = int((raw.loc[s:e, arr_cols] == 0).all(axis=1).sum())
    add_event("plant_zero_daytime", s, e, n, f"rows with all arrays zero: {allz}", "kept unchanged (candidate anomaly)")
for c in arr_cols:
    for s, e, n in runs(arr_out[c], min_len=ARR_MIN_RUN):
        add_event("array_outage", s, e, n, c, "kept unchanged (candidate anomaly)")

events_df = pd.DataFrame(events).sort_values("start_utc").reset_index(drop=True)
print(events_df["type"].value_counts().to_dict())
events_df.head(8)


plant-zero candidates (daytime, irr>100): 322 in 49 events
naive rule (irr>50 & plant<0.05, no sun-elevation filter): 10180 rows
array outage rows per array: {'arr1': 1337, 'arr2': 147, 'arr3': 2683, 'arr4': 670, 'arr5': 382, 'arr6': 255, 'arr7': 356, 'arr8': 309, 'arr9': 338, 'arr10': 269, 'arr11': 279, 'arr12': 137, 'arr13': 171, 'arr14': 125}


{'array_outage': 271, 'plant_zero_daytime': 49, 'weather_all_missing': 24, 'weather_partial_missing': 7, 'irradiance_sensor_fault': 3, 'module_temp_stuck': 1, 'temp_sensor_episode': 1, 'irradiance_offset': 1}


,type,start_utc,end_utc,rows,detail,action
0,plant_zero_daytime,2016-01-14 13:30:00+00:00,2016-01-14 13:30:00+00:00,1,rows with all arrays zero: 0,kept unchanged (candidate anomaly)
1,plant_zero_daytime,2016-03-08 15:30:00+00:00,2016-03-08 15:30:00+00:00,1,rows with all arrays zero: 0,kept unchanged (candidate anomaly)
2,plant_zero_daytime,2016-03-23 08:00:00+00:00,2016-03-23 08:15:00+00:00,2,rows with all arrays zero: 2,kept unchanged (candidate anomaly)
3,weather_all_missing,2016-03-23 08:00:00+00:00,2016-03-23 11:45:00+00:00,16,all 7 weather sensors missing,kept NaN (gap > fill limit)
4,weather_all_missing,2016-04-07 19:15:00+00:00,2016-04-09 17:15:00+00:00,185,all 7 weather sensors missing,kept NaN (gap > fill limit)
5,weather_all_missing,2016-04-27 04:30:00+00:00,2016-04-27 06:00:00+00:00,7,all 7 weather sensors missing,kept NaN (gap > fill limit)
6,plant_zero_daytime,2016-05-25 16:30:00+00:00,2016-05-25 16:45:00+00:00,2,rows with all arrays zero: 2,kept unchanged (candidate anomaly)
7,weather_all_missing,2016-05-25 16:30:00+00:00,2016-05-27 13:00:00+00:00,179,all 7 weather sensors missing,kept NaN (gap > fill limit)


## 7) Time split
Train through 2021, validation 2022, and test 2023. Validation and test start with 14 warm-up days; the prior 48 hours are marked in_purge_zone.


In [9]:
UTC = "UTC"
BOUNDS = {
    "train": (idx[0], pd.Timestamp("2021-12-31 23:45", tz=UTC)),
    "val": (pd.Timestamp("2022-01-01 00:00", tz=UTC), pd.Timestamp("2022-12-31 23:45", tz=UTC)),
    "test": (pd.Timestamp("2023-01-01 00:00", tz=UTC), idx[-1]),
}
split_label = pd.Series("", index=idx)
for name, (s, e) in BOUNDS.items():
    split_label[(idx >= s) & (idx <= e)] = name
assert (split_label != "").all()

in_purge = pd.Series(False, index=idx)
for nxt in ("val", "test"):
    st = BOUNDS[nxt][0]
    in_purge |= (idx >= st - pd.Timedelta(minutes=STEP_MIN * PURGE_ROWS)) & (idx < st)
print({k: int((split_label == k).sum()) for k in BOUNDS}, "| purge rows:", int(in_purge.sum()))


{'train': 210439, 'val': 35040, 'test': 35033} | purge rows: 384


## 8) Build groups


In [10]:
flags = pd.DataFrame(index=idx)
flags["solar_elev_deg"] = elev.round(3)
flags["is_night"], flags["is_day"] = is_night, is_day
for rule, m in rule_rows.items():
    flags[f"rule_{rule}"] = m
flags["wdir_wrapped"] = wrapped
flags["irr_offset_applied_wm2"] = offset_row.round(3)
for c, s in orig_vals.items():
    flags[f"orig_{c}"] = s
flags["wx_all_missing_raw"] = all_missing_raw
flags["wx_imputed_count"] = wx_imputed_count
flags["wx_unfilled_any"] = wx_unfilled_any
flags["wx_complete"] = ~wx_unfilled_any
flags["wx_complete_ex_irr"] = ~w[[c for c in wcols if c != "irr_wm2"]].isna().any(axis=1)
flags["cand_plant_zero_daytime"] = cand_zero
flags["array_outage_count"] = arr_out_count
for c in arr_cols:
    flags[f"array_outage_{c}"] = arr_out[c]
flags["split"] = split_label
flags["in_purge_zone"] = in_purge
flags["is_warmup"] = False
flags["scoreable_plant"] = plant.notna()
flags["scoreable_plant_with_weather"] = plant.notna() & ~wx_unfilled_any

groups = {
    "targets": raw[["plant_pu"]],
    "features_weather": w[weather_cols],
    "arrays": raw[arr_cols],
    "data_quality_flags": flags,
}
for g, f in groups.items():
    print(f"{g:20s} {f.shape}")


targets              (280512, 1)
features_weather     (280512, 9)
arrays               (280512, 14)
data_quality_flags   (280512, 40)


## 9) Fit scaler
Fit robust scaling on clean train rows only.


In [11]:
train_mask = split_label == "train"
scaler_cols = ["plant_pu"] + weather_cols + arr_cols
src = pd.concat([groups["targets"], groups["features_weather"], groups["arrays"]], axis=1)
params = {}
for c in scaler_cols:
    x = src.loc[train_mask, c].dropna()
    q25, med, q75 = (float(v) for v in x.quantile([0.25, 0.5, 0.75]))
    iqr, std = q75 - q25, float(x.std())
    scale, rule = (iqr, "iqr") if iqr > 0 else ((std, "std_fallback") if std > 0 else (1.0, "unit_fallback"))
    params[c] = {"median": med, "q25": q25, "q75": q75, "iqr": iqr, "std": std, "scale": scale,
                 "scale_rule": rule, "n_train_values": int(len(x))}
fit_sha = hashlib.sha256("|".join(idx[train_mask].strftime("%Y-%m-%dT%H:%M").tolist()).encode()).hexdigest()
scaler = {
    "method": "robust: (x - median) / scale",
    "fit_window": {"start_utc": str(idx[train_mask].min()), "end_utc": str(idx[train_mask].max()),
                   "rows": int(train_mask.sum()), "index_sha256": fit_sha},
    "applies_to": "plant_pu, cleaned weather, wind_u/wind_v, arrays. Features created later (lags/rolling) need their own train-only scaler.",
    "not_scaled": "flags, split labels, solar_elev_deg",
    "refit_note": "A final refit on train+val must be saved as a new versioned file with its own fit_window.",
    "params": params,
}
print("columns with IQR == 0:", [c for c, p in params.items() if p["scale_rule"] != "iqr"])


columns with IQR == 0: ['precip_mmh']


## 10) Write intermediate files


In [12]:
OUTPUTS = []


def to_csv_text(frame):
    f = frame.copy()
    for c in f.columns:
        if f[c].dtype == bool:
            f[c] = f[c].astype("int8")
    return f.to_csv(index_label="time_utc", float_format="%.4f")


def write_csv_gz(frame, name):
    with open(f"{OUT_DIR}/{name}", "wb") as raw_fh:
        with gzip.GzipFile(filename="", mode="wb", fileobj=raw_fh, mtime=0) as gz:
            gz.write(to_csv_text(frame).encode())
    OUTPUTS.append(name)


def make_split(frame, name, is_flags=False):
    s, e = BOUNDS[name]
    start = s if name == "train" else s - pd.Timedelta(minutes=STEP_MIN * WARMUP_ROWS)
    out = frame.loc[start:e].copy()
    if is_flags:
        out["is_warmup"] = out.index < s
        out["scoreable_plant"] = out["scoreable_plant"] & ~out["is_warmup"]
        out["scoreable_plant_with_weather"] = out["scoreable_plant_with_weather"] & ~out["is_warmup"]
    return out


for g, frame in groups.items():
    for name in BOUNDS:
        write_csv_gz(make_split(frame, name, is_flags=(g == "data_quality_flags")), f"{name}_{g}.csv.gz")

events_df.to_csv(f"{OUT_DIR}/events.csv", index=False); OUTPUTS.append("events.csv")
print(len(OUTPUTS), "files written")


13 files written


In [13]:
month_cov = {k: pd.Series(idx[split_label == k].tz_convert(LOCAL_TZ).month).value_counts().sort_index().to_dict() for k in BOUNDS}
quarters = pd.date_range("2021-01-01", "2022-10-01", freq="QS", tz=UTC)
cv_folds = []
for i, q in enumerate(quarters, 1):
    v_end = (q + pd.offsets.QuarterEnd(0)).normalize() + pd.Timedelta(hours=23, minutes=45)
    cv_folds.append({"fold": i, "train_first_origin": str(idx[0]),
                     "train_last_origin": str(q - pd.Timedelta(minutes=STEP_MIN * (PURGE_ROWS + 1))),
                     "val_start": str(q), "val_end": str(v_end)})
known_issues = {k: [] for k in BOUNDS}
for _, ev_ in events_df[events_df["type"].isin(["irradiance_sensor_fault", "irradiance_offset", "temp_sensor_episode",
                                                  "module_temp_stuck", "weather_all_missing"])].iterrows():
    if ev_["type"] == "weather_all_missing" and ev_["rows"] < 96:
        continue
    for k, (bs, be) in BOUNDS.items():
        a, b = max(ev_["start_utc"], bs), min(ev_["end_utc"], be)
        if a <= b:
            known_issues[k].append(f"{ev_['type']}: {a.date()} .. {b.date()}")
splits = {
    "timezone": "UTC", "step_minutes": STEP_MIN,
    "bounds": {k: {"start": str(s), "end": str(e), "rows": int((split_label == k).sum())} for k, (s, e) in BOUNDS.items()},
    "warmup_rows": WARMUP_ROWS, "purge_rows": PURGE_ROWS,
    "rows_per_day": 96, "horizon_rows": {"24h": 96, "48h": 192},
    "rows_in_files_including_warmup": {k: int(len(make_split(groups["targets"], k))) for k in BOUNDS},
    "how_to_use": ("val_*/test_* files start with warm-up rows (is_warmup=1): use them only to build lags/rolling features, "
                   "never for training or scoring. Drop forecast origins flagged in_purge_zone when building training samples."),
    "months_coverage_rows_athens_time": month_cov,
    "known_data_issues_by_split": known_issues,
    "optional_rolling_origin_cv": cv_folds,
}
dump_json(splits, f"{OUT_DIR}/splits.json"); OUTPUTS.append("splits.json")
dump_json(scaler, f"{OUT_DIR}/scaler_params.json"); OUTPUTS.append("scaler_params.json")

column_groups = {
    "targets": {"columns": ["plant_pu"], "role": "label: normalised plant active power"},
    "features_weather": {"columns": weather_cols, "role": "OBSERVED weather (same time step)"},
    "arrays": {"columns": arr_cols, "role": "same-step outcomes per array: anomaly evidence only, NOT features"},
    "data_quality_flags": {"columns": list(flags.columns), "role": "flags, masks, original values, split labels"},
}
availability = {
    "forecast_protocol_ASSUMPTION": {
        "note": "Default protocol proposed during preprocessing; change it deliberately before feature engineering.",
        "forecast_origin": "every day at 00:00 UTC",
        "horizons_rows": "1..192 after the origin (96 = 24h, 192 = 48h; report both)",
        "known_at_origin": ["plant and array history up to the origin", "observed weather up to the origin", "deterministic calendar and solar geometry"],
        "not_known_at_origin": ["any weather value inside the forecast window (the dataset has no weather forecasts)"],
    },
    "columns": {
        "targets": "history only: lags >= horizon",
        "features_weather": "observed, not forecast: lag by >= horizon, or use declared weather scenarios",
        "arrays": "same-step outcomes: forbidden as forecasting features (lags only)",
        "data_quality_flags": "never a model feature; use for masking/evaluation only (is_night/solar_elev_deg are deterministic and may be recomputed as features)",
    },
}
dump_json(column_groups, f"{OUT_DIR}/column_groups.json"); OUTPUTS.append("column_groups.json")
dump_json(availability, f"{OUT_DIR}/feature_availability.json"); OUTPUTS.append("feature_availability.json")
print(len(OUTPUTS), "artifacts written")


17 artifacts written


## 11) Build modeling tables


In [14]:
SPLITS = ("train", "val", "test")
GROUPS = ("targets", "features_weather", "arrays", "data_quality_flags")
EXPECTED_ROWS = {"train": 210439, "val": 35040, "test": 35033}
STEP = pd.Timedelta("15min")
MARGIN_ROWS = 4
FEATURES = ["irr_wm2", "temp_c", "mod_temp_c", "solar_elev_deg", "hour_sin", "hour_cos", "doy_sin", "doy_cos"]
TARGET = "plant_pu"
AUX = ["is_day", "wx_complete", "cand_plant_zero_daytime", "array_outage_count",
       "rule_modt_stuck", "rule_temp_sensor_episode", "rule_irr_fault_window"]
REQUIRED = {"features_weather": ["irr_wm2", "temp_c", "mod_temp_c"], "targets": [TARGET], "data_quality_flags": ["solar_elev_deg", "is_warmup"] + AUX}
IN_DIR = OUT_DIR


## A1. Load and trim warm-up


In [15]:
rd = lambda n: pd.read_csv(f"{IN_DIR}/{n}", index_col="time_utc", parse_dates=True)
raw = {s: {g: rd(f"{s}_{g}.csv.gz") for g in GROUPS} for s in SPLITS}

 # column check
for g in GROUPS:
    cols = list(raw["train"][g].columns)
    assert all(list(raw[s][g].columns) == cols for s in SPLITS), f"columns differ between splits in {g}"
    print(f"{g} ({len(cols)}):", cols)
for g, cols in REQUIRED.items():
    absent = [c for c in cols if c not in raw["train"][g].columns]
    assert not absent, f"columns missing in {g}: {absent}"

 # matching indexes
for s in SPLITS:
    ref = raw[s]["data_quality_flags"].index
    assert all((raw[s][g].index == ref).all() for g in GROUPS), f"index differs between files in {s}"
    keep = ref[raw[s]["data_quality_flags"]["is_warmup"].to_numpy() == 0]
    n_warm = len(ref) - len(keep)
    for g in GROUPS:
        raw[s][g] = raw[s][g].loc[keep]
    print(f"{s}: rows before {len(ref):,} | warm-up removed {n_warm:,} | after {len(keep):,}")

for s in SPLITS:
    idxs = [raw[s][g].index for g in GROUPS]
    assert all(i.is_unique for i in idxs)
    assert all((i == idxs[0]).all() for i in idxs)
    assert idxs[0].is_monotonic_increasing
    assert (idxs[0].to_series().diff().dropna() == STEP).all(), f"step is not 15 min in {s}"
    assert len(idxs[0]) == EXPECTED_ROWS[s], f"unexpected row count in {s}: {len(idxs[0])} != {EXPECTED_ROWS[s]}"
print("A1 asserts passed:", {s: len(raw[s]['targets']) for s in SPLITS})


targets (1): ['plant_pu']
features_weather (9): ['temp_c', 'irr_wm2', 'mod_temp_c', 'precip_mmh', 'rh_pct', 'wdir_deg', 'wspd_ms', 'wind_u', 'wind_v']
arrays (14): ['arr1', 'arr2', 'arr3', 'arr4', 'arr5', 'arr6', 'arr7', 'arr8', 'arr9', 'arr10', 'arr11', 'arr12', 'arr13', 'arr14']
data_quality_flags (40): ['solar_elev_deg', 'is_night', 'is_day', 'rule_wspd_gt_40', 'rule_temp_sensor_episode', 'rule_modt_stuck', 'rule_irr_fault_window', 'rule_irr_gt_1400', 'wdir_wrapped', 'irr_offset_applied_wm2', 'orig_temp_c', 'orig_irr_wm2', 'orig_mod_temp_c', 'orig_wspd_ms', 'wx_all_missing_raw', 'wx_imputed_count', 'wx_unfilled_any', 'wx_complete', 'wx_complete_ex_irr', 'cand_plant_zero_daytime', 'array_outage_count', 'array_outage_arr1', 'array_outage_arr2', 'array_outage_arr3', 'array_outage_arr4', 'array_outage_arr5', 'array_outage_arr6', 'array_outage_arr7', 'array_outage_arr8', 'array_outage_arr9', 'array_outage_arr10', 'array_outage_arr11', 'array_outage_arr12', 'array_outage_arr13', 'array_ou

## A2. Modeling table
Build the eight modeling features and keep flags separate.


In [16]:
def build_table(s):
    t, w, f = raw[s]["targets"], raw[s]["features_weather"], raw[s]["data_quality_flags"]
    idx = t.index
    hour = idx.hour.to_numpy() + idx.minute.to_numpy() / 60.0
    doy = idx.day_of_year.to_numpy()
    tab = pd.DataFrame({
        "irr_wm2": w["irr_wm2"], "temp_c": w["temp_c"], "mod_temp_c": w["mod_temp_c"],
        "solar_elev_deg": f["solar_elev_deg"],
        "hour_sin": np.sin(2 * np.pi * hour / 24.0), "hour_cos": np.cos(2 * np.pi * hour / 24.0),
        "doy_sin": np.sin(2 * np.pi * doy / 365.25), "doy_cos": np.cos(2 * np.pi * doy / 365.25),
        TARGET: t[TARGET],
    }, index=idx)
    for c in AUX:
        tab[c] = f[c]
    return tab

tables = {s: build_table(s) for s in SPLITS}
assert list(tables["train"].columns[:8]) == FEATURES
forbidden = [c for c in FEATURES if c == TARGET or c.startswith("arr") or c.startswith(("rule_", "cand_", "wx_", "is_"))]
assert not forbidden and len(FEATURES) == 8
print("model inputs (8):", FEATURES)
print("target:", TARGET, "| aux (not inputs):", AUX)
for s in SPLITS:
    assert (tables[s]["is_day"].to_numpy() == (tables[s]["solar_elev_deg"].to_numpy() > 10).astype(int)).all()
print("A2 asserts passed (is_day == solar_elev_deg > 10 in all splits)")


model inputs (8): ['irr_wm2', 'temp_c', 'mod_temp_c', 'solar_elev_deg', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos']
target: plant_pu | aux (not inputs): ['is_day', 'wx_complete', 'cand_plant_zero_daytime', 'array_outage_count', 'rule_modt_stuck', 'rule_temp_sensor_episode', 'rule_irr_fault_window']
A2 asserts passed (is_day == solar_elev_deg > 10 in all splits)


## A3. Scoreable rows


In [17]:
for s in SPLITS:
    t = tables[s]
    t["scoreable"] = ((t["is_day"] == 1) & (t["wx_complete"] == 1)
                      & t[FEATURES].notna().all(axis=1) & t[TARGET].notna()).astype(int)

def cov_row(label, t):
    d, sc = int((t["is_day"] == 1).sum()), int(t["scoreable"].sum())
    return {"split": label, "rows": len(t), "is_day rows": d, "scoreable rows": sc, "coverage %": round(100 * sc / d, 2)}

rows = [cov_row(s, tables[s]) for s in SPLITS]
CUT = pd.Timestamp("2023-10-12", tz="UTC")
tt = tables["test"]
rows += [cov_row("test (< 2023-10-12)", tt[tt.index < CUT]), cov_row("test (>= 2023-10-12)", tt[tt.index >= CUT])]
a3 = pd.DataFrame(rows)
print(a3.to_string(index=False))


               split   rows  is_day rows  scoreable rows  coverage %
               train 210439        88943           86985       97.80
                 val  35040        14815           12214       82.44
                test  35033        14812           12180       82.23
 test (< 2023-10-12)  27264        12295           12180       99.06
test (>= 2023-10-12)   7769         2517               0        0.00


## A4. Clean set


In [18]:
ev = pd.read_csv(f"{IN_DIR}/events.csv")
ev["start_utc"] = pd.to_datetime(ev["start_utc"], utc=True)
ev["end_utc"] = pd.to_datetime(ev["end_utc"], utc=True)
real = ev[ev["type"].isin(["plant_zero_daytime", "array_outage"])].reset_index(drop=True)
print("events in file:", ev["type"].value_counts().to_dict())
print("events used for margins:", real["type"].value_counts().to_dict())

def mark_margin(index, events, margin_rows=MARGIN_ROWS):
    vals = index.asi8
    lo = (events["start_utc"] - margin_rows * STEP).astype("int64").to_numpy()
    hi = (events["end_utc"] + margin_rows * STEP).astype("int64").to_numpy()
 # matching time units
    unit = index.dtype.unit
    scale = {"ns": 1, "us": 1000, "ms": 10**6, "s": 10**9}[unit]
    a = np.searchsorted(vals, (events["start_utc"] - margin_rows * STEP).map(lambda x: x.value // scale).to_numpy(), side="left")
    b = np.searchsorted(vals, (events["end_utc"] + margin_rows * STEP).map(lambda x: x.value // scale).to_numpy(), side="right")
    delta = np.zeros(len(index) + 1, dtype=int)
    np.add.at(delta, a, 1); np.add.at(delta, b, -1)
    return (np.cumsum(delta)[:-1] > 0)

for s in SPLITS:
    t = tables[s]
    t["in_event_margin"] = mark_margin(t.index, real).astype(int)
    t["clean"] = ((t["scoreable"] == 1) & (t["cand_plant_zero_daytime"] == 0) & (t["array_outage_count"] == 0)
                  & (t["rule_modt_stuck"] == 0) & (t["rule_temp_sensor_episode"] == 0)
                  & (t["rule_irr_fault_window"] == 0) & (t["in_event_margin"] == 0)).astype(int)
    t["split"] = s

a4 = pd.DataFrame([{"split": s, "scoreable": int(tables[s]["scoreable"].sum()), "in_event_margin rows": int(tables[s]["in_event_margin"].sum()),
                    "clean rows": int(tables[s]["clean"].sum())} for s in SPLITS])
print(a4.to_string(index=False))
assert tables["train"]["clean"].sum() >= 20000, "STOP: train clean < 20,000"
assert tables["val"]["clean"].sum() >= 5000, "STOP: val clean < 5,000"
assert all(((tables[s]["clean"] == 1) <= (tables[s]["scoreable"] == 1)).all() for s in SPLITS)

 # correction period
OFF_A, OFF_B = pd.Timestamp("2022-08-21", tz="UTC"), pd.Timestamp("2023-10-12", tz="UTC")
offset_clean = {s: int(tables[s]["clean"][(tables[s].index >= OFF_A) & (tables[s].index < OFF_B)].sum()) for s in SPLITS}
print("clean rows inside the irradiance-offset correction period (2022-08-21 .. 2023-10-11):", offset_clean)


events in file: {'array_outage': 271, 'plant_zero_daytime': 49, 'weather_all_missing': 24, 'weather_partial_missing': 7, 'irradiance_sensor_fault': 3, 'module_temp_stuck': 1, 'temp_sensor_episode': 1, 'irradiance_offset': 1}
events used for margins: {'array_outage': 271, 'plant_zero_daytime': 49}
split  scoreable  in_event_margin rows  clean rows
train      86985                  3653       83135
  val      12214                   121       12146
 test      12180                  3356        8857
clean rows inside the irradiance-offset correction period (2022-08-21 .. 2023-10-11): {'train': 0, 'val': 4659, 'test': 8857}


In [19]:
 # fit on clean train only
tr = tables["train"]; trc = tr[tr["clean"] == 1]
params = {}
for c in FEATURES + [TARGET]:
    x = trc[c].dropna()
    q25, med, q75 = (float(v) for v in x.quantile([0.25, 0.5, 0.75]))
    iqr, std = q75 - q25, float(x.std())
    scale, rule = (iqr, "iqr") if iqr > 0 else (std, "std_fallback")
    params[c] = {"median": med, "q25": q25, "q75": q75, "iqr": iqr, "std": std, "scale": scale, "scale_rule": rule}
    assert scale > 0
scaler_clean = {"method": "(x - median) / scale", "fit_set": "clean rows of train only",
                "fit_rows": int(len(trc)), "fit_window": {"start_utc": str(trc.index.min()), "end_utc": str(trc.index.max())},
                "columns": FEATURES + [TARGET], "params": params}
dump_json(scaler_clean, f"{OUT_DIR}/scaler_clean.json")
print(pd.DataFrame(params).T[["median", "iqr", "std", "scale", "scale_rule"]].round(5).to_string())


                  median       iqr         std     scale scale_rule
irr_wm2         508.5056  546.9669  303.060369  546.9669        iqr
temp_c           19.7483   10.0779    6.188839   10.0779        iqr
mod_temp_c       26.8559  14.02445    9.154289  14.02445        iqr
solar_elev_deg    32.453   25.8665   16.658334   25.8665        iqr
hour_sin        0.321439  1.187465    0.624227  1.187465        iqr
hour_cos       -0.793353  0.504641     0.34589  0.504641        iqr
doy_sin         0.096613  1.430938    0.708724  1.430938        iqr
doy_cos        -0.283139  1.308668    0.688369  1.308668        iqr
plant_pu          0.4345    0.4501    0.244521    0.4501        iqr


## A5. Synthetic fault manifest


In [20]:
tt = tables["test"].copy()
assert (tt.index.to_series().diff().dropna() == STEP).all()
tt["day"] = tt.index.normalize()
daymask = tt["solar_elev_deg"] > 10
tt["ok"] = ((tt["scoreable"] == 1) & daymask).to_numpy()

g = tt.groupby("day")
day_tbl = pd.DataFrame({
    "rows": g.size(),
    "event_rows": g["in_event_margin"].sum(),
    "day_rows": daymask.groupby(tt["day"]).sum(),
    "scoreable_day_rows": (tt["scoreable"].astype(bool) & daymask).groupby(tt["day"]).sum(),
})
day_tbl["coverage"] = (day_tbl["scoreable_day_rows"] / day_tbl["day_rows"]).where(day_tbl["day_rows"] > 0)
day_tbl["no_event"] = day_tbl["event_rows"] == 0
day_tbl["cov_ok"] = day_tbl["coverage"] >= 0.8
day_tbl["eligible"] = day_tbl["no_event"] & day_tbl["cov_ok"]
print("test UTC days:", len(day_tbl), "| no real event:", int(day_tbl["no_event"].sum()),
      "| coverage >= 80%:", int(day_tbl["cov_ok"].sum()), "| eligible (both):", int(day_tbl["eligible"].sum()))

DURS = {1: 4, 3: 12, 6: 24} # hours -> rows
ok_by_day = {d: x["ok"].to_numpy() for d, x in tt.groupby("day")}
idx_by_day = {d: x.index for d, x in tt.groupby("day")}

def feasible_starts(ok, n):
    if len(ok) < n:
        return np.array([], dtype=int)
    cs = np.concatenate([[0], np.cumsum(ok.astype(int))])
    return np.flatnonzero((cs[n:] - cs[:-n]) == n)

elig_days = list(day_tbl.index[day_tbl["eligible"]])
cap_rows = []
for h, n in DURS.items():
    able = [d for d in elig_days if len(feasible_starts(ok_by_day[d], n)) > 0]
    cap_rows.append({"duration_h": h, "eligible days (total)": len(elig_days), "days that fit the event": len(able)})
cap = pd.DataFrame(cap_rows)
print(cap.to_string(index=False))


test UTC days: 365 | no real event: 279 | coverage >= 80%: 280 | eligible (both): 199
 duration_h  eligible days (total)  days that fit the event
          1                    199                      199
          3                    199                      199
          6                    199                      199


In [21]:
CELLS = [("drop", s, h) for s in (0.10, 0.20, 0.40, 1.00) for h in (1, 3, 6)] + [("stuck", np.nan, h) for h in (1, 3, 6)]
N_PER_CELL = 25
test_plant = tt[TARGET].to_numpy()

def make_manifest():
    recs, starts_log = [], {}
    for ci, (typ, sev, h) in enumerate(CELLS):
        n = DURS[h]
        rng = np.random.default_rng([SEED, ci])
        able = [d for d in elig_days if len(feasible_starts(ok_by_day[d], n)) > 0]
        k = min(N_PER_CELL, len(able))
        chosen = sorted(rng.choice(len(able), size=k, replace=False)) if k > 0 else []
        for j in chosen:
            d = able[j]
            fs = feasible_starts(ok_by_day[d], n)
            st = int(rng.choice(fs))
            ix = idx_by_day[d]
            sev_tag = "NA" if typ == "stuck" else f"s{int(round(sev * 100)):03d}"
            recs.append({"event_id": f"{typ}_{sev_tag}_{h}h_{d.date()}", "type": typ, "severity": sev, "duration_h": h,
                         "day": str(d.date()), "start_utc": ix[st], "end_utc": ix[st + n - 1]})
    return pd.DataFrame(recs)

manifest = make_manifest()
assert manifest["event_id"].is_unique
assert set(zip(manifest["type"], manifest["severity"].fillna(-1), manifest["duration_h"])) <= set((a, -1 if np.isnan(b) else b, c) for a, b, c in CELLS)

 # valid event rows
pos = {t: i for i, t in enumerate(tt.index)}
for r in manifest.itertuples():
    a, b = pos[r.start_utc], pos[r.end_utc]
    seg = tt.iloc[a:b + 1]
    assert len(seg) == DURS[r.duration_h] and seg["ok"].all() and (seg["in_event_margin"] == 0).all()
    assert seg.index[0].normalize() == seg.index[-1].normalize()

per_cell = (manifest.assign(sev=manifest["severity"].fillna(-1)).groupby(["type", "sev", "duration_h"]).size()
            .rename("events").reset_index())
per_cell["sev"] = per_cell["sev"].replace(-1, np.nan)
print(per_cell.to_string(index=False))
print("total events:", len(manifest), "| cells:", len(per_cell), "| max per day-cell pair:",
      int(manifest.groupby(["type", "severity", "duration_h", "day"], dropna=False).size().max()))
manifest.to_csv(f"{OUT_DIR}/inject_manifest.csv", index=False,
                columns=["event_id", "type", "severity", "duration_h", "day", "start_utc", "end_utc"])


 type  sev  duration_h  events
 drop  0.1           1      25
 drop  0.1           3      25
 drop  0.1           6      25
 drop  0.2           1      25
 drop  0.2           3      25
 drop  0.2           6      25
 drop  0.4           1      25
 drop  0.4           3      25
 drop  0.4           6      25
 drop  1.0           1      25
 drop  1.0           3      25
 drop  1.0           6      25
stuck  NaN           1      25
stuck  NaN           3      25
stuck  NaN           6      25
total events: 375 | cells: 15 | max per day-cell pair: 1


In [22]:
 # event size
changes = []
for r in manifest.itertuples():
    a, b = pos[r.start_utc], pos[r.end_utc]
    seg = test_plant[a:b + 1]
    delta = seg * r.severity if r.type == "drop" else np.abs(seg - seg[0])
    changes.append(float(np.mean(np.abs(delta))))
manifest_chg = manifest.assign(sev=manifest["severity"].fillna(-1), mean_abs_change_pu=changes)
chg = (manifest_chg.groupby(["type", "sev", "duration_h"])["mean_abs_change_pu"].agg(["count", "median"]).round(4).reset_index())
chg["sev"] = chg["sev"].replace(-1, np.nan)
print(chg.rename(columns={"count": "events", "median": "median mean_abs_change_pu"}).to_string(index=False))


 type  sev  duration_h  events  median mean_abs_change_pu
 drop  0.1           1      25                     0.0408
 drop  0.1           3      25                     0.0566
 drop  0.1           6      25                     0.0429
 drop  0.2           1      25                     0.0867
 drop  0.2           3      25                     0.1087
 drop  0.2           6      25                     0.1008
 drop  0.4           1      25                     0.2163
 drop  0.4           3      25                     0.1897
 drop  0.4           6      25                     0.2077
 drop  1.0           1      25                     0.4768
 drop  1.0           3      25                     0.5324
 drop  1.0           6      25                     0.5702
stuck  NaN           1      25                     0.0632
stuck  NaN           3      25                     0.1491
stuck  NaN           6      25                     0.1782


## A6. Final outputs


In [23]:
OUT_COLS = FEATURES + [TARGET, "scoreable", "clean", "in_event_margin", "split"]
def write_table(s):
    t = tables[s][OUT_COLS]
    txt = t.to_csv(index_label="time_utc", float_format="%.6f")
    with open(f"{OUT_DIR}/table_{s}.csv.gz", "wb") as fh:
        with gzip.GzipFile(filename="", mode="wb", fileobj=fh, mtime=0) as gz:
            gz.write(txt.encode())
for s in SPLITS:
    write_table(s)

summary = {
    "environment": {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__, "seed": SEED},
    "features": FEATURES, "target": TARGET, "output_columns": OUT_COLS,
    "rows_after_warmup_removal": {s: len(tables[s]) for s in SPLITS},
    "coverage_table": a3.to_dict(orient="records"),
    "clean_table": a4.to_dict(orient="records"),
    "events_used_for_margins": real["type"].value_counts().to_dict(), "margin_rows_each_side": MARGIN_ROWS,
    "scaler_fit_rows": int(len(trc)),
    "test_days": {"total": int(len(day_tbl)), "no_real_event": int(day_tbl["no_event"].sum()),
                  "coverage_ge_80": int(day_tbl["cov_ok"].sum()), "eligible": int(day_tbl["eligible"].sum())},
    "eligible_days_by_duration": cap.to_dict(orient="records"),
    "injection_cells": per_cell.assign(sev=per_cell["sev"].astype(object).where(per_cell["sev"].notna(), None)).to_dict(orient="records"),
    "injection_events_total": int(len(manifest)),
    "injection_change_size_pu": chg.assign(sev=chg["sev"].astype(object).where(chg["sev"].notna(), None)).to_dict(orient="records"),
    "notes": [
        "Irradiance-offset correction period (2022-08-21 .. 2023-10-11) stays inside clean; clean rows inside it: " + json.dumps(offset_clean),
        "in_event_margin marks every row inside plant_zero_daytime/array_outage events expanded by 4 rows on each side (event rows included).",
        "hour/doy features use UTC; doy period 365.25 days.",
        "Injection start inside a day is random among feasible positions (default_rng([42, cell_index])); severity is blank for stuck.",
        "Scaler uses clean train rows only; scaler_params.json from the preprocessing step is not used.",
    ],
}
dump_json(summary, f"{OUT_DIR}/dataset_summary.json")

 # final check
for s in SPLITS:
    t = pd.read_csv(f"{OUT_DIR}/table_{s}.csv.gz", index_col="time_utc", parse_dates=True)
    assert len(t) == EXPECTED_ROWS[s] and list(t.columns) == OUT_COLS
    assert (t["split"] == s).all() and t.index.is_unique
    assert ((t["clean"] == 1) <= (t["scoreable"] == 1)).all()
    assert t.loc[t["scoreable"] == 1, FEATURES + [TARGET]].notna().all().all()
m = pd.read_csv(f"{OUT_DIR}/inject_manifest.csv")
assert list(m.columns) == ["event_id", "type", "severity", "duration_h", "day", "start_utc", "end_utc"] and len(m) == len(manifest)
print("A6 asserts passed")
print("dataset_summary.json", f"{os.path.getsize(f'{OUT_DIR}/dataset_summary.json'):,d}", "bytes")


A6 asserts passed
dataset_summary.json 6,307 bytes


## 12) Quick check


In [24]:
# this is slow, but works
for split in SPLITS:
    df = pd.read_csv(f"{OUT_DIR}/table_{split}.csv.gz", index_col="time_utc", parse_dates=True)
    print(split, df.shape, list(df.columns))
assert len(manifest) == 375
assert all(len(pd.read_csv(f"{OUT_DIR}/table_{s}.csv.gz")) == EXPECTED_ROWS[s] for s in SPLITS)
print("done")


train (210439, 13) ['irr_wm2', 'temp_c', 'mod_temp_c', 'solar_elev_deg', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'plant_pu', 'scoreable', 'clean', 'in_event_margin', 'split']
val (35040, 13) ['irr_wm2', 'temp_c', 'mod_temp_c', 'solar_elev_deg', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'plant_pu', 'scoreable', 'clean', 'in_event_margin', 'split']


test (35033, 13) ['irr_wm2', 'temp_c', 'mod_temp_c', 'solar_elev_deg', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'plant_pu', 'scoreable', 'clean', 'in_event_margin', 'split']


done
